# 📓 Distributed Image Filtering with Dask, Ray, and dask-sql

**Overview**:
This notebook demonstrates how to load and analyze large-scale image attribute data stored in Parquet files (located in `image_filter_pipeline/data/processed`) using **Dask** for parallel processing and **Ray** as the distributed scheduler. We also leverage **dask-sql** for SQL-like queries and outline additional use cases for AI researchers.

**Data Schema**:
The Parquet files have the following columns:
`["", "image_url", "face_confidence", "bbox", "glasses_label", "glasses_confidence", "clip_metadata", "clip_embedding"]`

*Note*: The first column is unnamed (empty string) and can be ignored if not needed.



In [3]:
from ray_dask_init import initialize_ray_and_dask

# 🔧 Initialization & Setup

initialize_ray_and_dask()



2025-03-24 13:28:18,319	INFO worker.py:1774 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 


Dask client initialized. Dashboard available at: http://127.0.0.1:56338/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:56338/status,
Dashboard: http://127.0.0.1:56338/status,Workers: 4
Total threads: 8,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:56339,Workers: 4
Dashboard: http://127.0.0.1:56338/status,Total threads: 8
Started: Just now,Total memory: 29.80 GiB
Comm: tcp://127.0.0.1:56355,Total threads: 2
Dashboard: http://127.0.0.1:56359/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:56342,


## 📂 Load Parquet Data into Dask

Adjust the `parquet_path` to point to your dataset. The following code loads the Parquet files into a Dask DataFrame and inspects the schema.


In [4]:
import pathlib
import dask.dataframe as dd

# Print current working directory for debugging
current_dir = pathlib.Path.cwd()
print("Current working directory:", current_dir)

# If the current directory contains "notebooks", assume the project root is one level up.
if "notebooks" in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

# Construct the correct path to the Parquet file
parquet_file = project_root / "data" / "processed" / "part-0.snappy.parquet"
print("Adjusted parquet file path:", parquet_file)

# Read the Parquet file using only the desired columns.
df = dd.read_parquet(
    str(parquet_file),  # Convert Path to str
    engine="pyarrow",
)

# Display a preview of the dataset
print("Preview of the dataset:")
df.head()


Current working directory: /Users/bikash/stability-ai/version1/image_filter_pipeline/notebooks
Adjusted parquet file path: /Users/bikash/stability-ai/version1/image_filter_pipeline/data/processed/part-0.snappy.parquet
Preview of the dataset:


,image_url,face_confidence,bbox,glasses_label,glasses_confidence,clip_metadata,clip_embedding
0,https://upload.wikimedia.org/wikipedia/commons...,0.893632,"[114, 63, 223, 165]",A person wearing reading glasses,0.681879,"{""A person wearing reading glasses"": 0.6818788...","[0.003890973, -0.024154034, -0.008553939, 0.00..."
1,https://upload.wikimedia.org/wikipedia/commons...,0.571500,"[21, 34, 277, 320]",no confident label,0.420454,"{""A person wearing reading glasses"": 0.2502837...","[-0.041819025, 0.0032307282, 0.014172979, -0.0..."
2,https://upload.wikimedia.org/wikipedia/commons...,0.832178,"[127, 78, 182, 133]",A person wearing no glasses,0.953353,"{""A person wearing reading glasses"": 0.0212386...","[-0.01806082, 0.0012981347, 0.0010393162, -0.0..."
3,https://upload.wikimedia.org/wikipedia/commons...,0.873145,"[140, 32, 196, 124]",A person wearing no glasses,0.829759,"{""A person wearing reading glasses"": 0.1326903...","[0.031326413, -0.04205955, -0.056005336, -0.00..."
4,https://upload.wikimedia.org/wikipedia/commons...,0.865377,"[177, 91, 228, 170]",A person wearing no glasses,0.914516,"{""A person wearing reading glasses"": 0.0356861...","[-0.005093807, -0.007854619, -0.03761017, 0.01..."


## 😎 Filter Example 1: Images with High Glasses Confidence

Retrieve images where the `glasses_confidence` exceeds a specified threshold.



In [5]:
# Define the glasses confidence threshold
GLASSES_CONFIDENCE_THRESHOLD = 0.8

# Filter images based on the glasses_confidence column
glasses_df = df[df['glasses_confidence'] > GLASSES_CONFIDENCE_THRESHOLD]

# Compute and display the filtered results
result_glasses = glasses_df.compute()
print(f"Total images with glasses_confidence > {GLASSES_CONFIDENCE_THRESHOLD}: {len(result_glasses)}")
result_glasses.head()


Total images with glasses_confidence > 0.8: 2055


,image_url,face_confidence,bbox,glasses_label,glasses_confidence,clip_metadata,clip_embedding
2,https://upload.wikimedia.org/wikipedia/commons...,0.832178,"[127, 78, 182, 133]",A person wearing no glasses,0.953353,"{""A person wearing reading glasses"": 0.0212386...","[-0.01806082, 0.0012981347, 0.0010393162, -0.0..."
3,https://upload.wikimedia.org/wikipedia/commons...,0.873145,"[140, 32, 196, 124]",A person wearing no glasses,0.829759,"{""A person wearing reading glasses"": 0.1326903...","[0.031326413, -0.04205955, -0.056005336, -0.00..."
4,https://upload.wikimedia.org/wikipedia/commons...,0.865377,"[177, 91, 228, 170]",A person wearing no glasses,0.914516,"{""A person wearing reading glasses"": 0.0356861...","[-0.005093807, -0.007854619, -0.03761017, 0.01..."
7,https://upload.wikimedia.org/wikipedia/commons...,0.837970,"[116, 60, 211, 156]",A person wearing no glasses,0.957388,"{""A person wearing reading glasses"": 0.0305605...","[-0.07292417, -0.0046823695, -0.0144624105, 0...."
10,https://upload.wikimedia.org/wikipedia/commons...,0.791787,"[121, 14, 179, 74]",A person wearing no glasses,0.875261,"{""A person wearing reading glasses"": 0.0779443...","[0.009621351, -0.022147348, -0.0028696575, -0...."


## 😃 Filter Example 2: Nested Filters for Face and Glasses Confidence

Retrieve images that meet two conditions:
- **face_confidence** exceeds a given threshold.
- **glasses_confidence** exceeds a given threshold.


In [6]:
# Define thresholds for face and glasses confidence
FACE_CONFIDENCE_THRESHOLD = 0.9
GLASSES_CONFIDENCE_THRESHOLD = 0.7

# Apply nested filtering conditions on the DataFrame
nested_filter_df = df[
    (df['face_confidence'] > FACE_CONFIDENCE_THRESHOLD) &
    (df['glasses_confidence'] > GLASSES_CONFIDENCE_THRESHOLD)
]

# Compute and display the nested filtered results
nested_result = nested_filter_df.compute()
print(f"Total images meeting both face and glasses confidence thresholds: {len(nested_result)}")
nested_result.head()


Total images meeting both face and glasses confidence thresholds: 23


,image_url,face_confidence,bbox,glasses_label,glasses_confidence,clip_metadata,clip_embedding
165,https://upload.wikimedia.org/wikipedia/commons...,0.902625,"[121, 51, 196, 174]",A person wearing reading glasses,0.724856,"{""A person wearing reading glasses"": 0.7248564...","[0.025833458, -0.0430649, -0.017536689, -0.001..."
196,http://upload.wikimedia.org/wikipedia/commons/...,0.906267,"[127, 43, 237, 165]",A person wearing no glasses,0.785007,"{""A person wearing reading glasses"": 0.0391257...","[0.013481831, 0.0009165322, -0.0012811162, 0.0..."
303,https://upload.wikimedia.org/wikipedia/commons...,0.910249,"[94, 59, 182, 205]",A person wearing no glasses,0.834284,"{""A person wearing reading glasses"": 0.0392256...","[-0.018733094, -0.05125668, -0.01450398, 0.000..."
481,https://upload.wikimedia.org/wikipedia/commons...,0.900473,"[0, 36, 65, 170]",A person wearing no glasses,0.833429,"{""A person wearing reading glasses"": 0.0991516...","[0.01877181, -0.013439487, -0.024779111, 0.004..."
548,https://upload.wikimedia.org/wikipedia/commons...,0.901883,"[105, 57, 212, 176]",A person wearing reading glasses,0.727224,"{""A person wearing reading glasses"": 0.7272238...","[-0.022703255, -0.0075476426, -0.017506437, 0...."


## 🎚️ Advanced Nested Filtering Example

Retrieve images where:
- **face_confidence** is high (e.g., > 0.X) **AND**
- Either **glasses_confidence** is high (e.g., > 0.8) or the `glasses_label` explicitly indicates 'sunglasses'.

*Note*: Adjust the condition on `glasses_label` based on your dataset’s specific values.


In [15]:
# Advanced filtering: high face confidence AND either high glasses confidence OR a specific glasses label
advanced_nested_df = df[
    (df['face_confidence'] > 0.8) & (
        (df['glasses_confidence'] > 0.8) | (df['glasses_label'] == 'sunglasses')
    )
]

# Compute and display the advanced filtering results
advanced_result = advanced_nested_df.compute()
print(f"Total images matching advanced nested conditions: {len(advanced_result)}")
advanced_result.head()


Total images matching advanced nested conditions: 1587


,image_url,face_confidence,bbox,glasses_label,glasses_confidence,clip_metadata,clip_embedding
2,https://upload.wikimedia.org/wikipedia/commons...,0.832178,"[127, 78, 182, 133]",A person wearing no glasses,0.953353,"{""A person wearing reading glasses"": 0.0212386...","[-0.01806082, 0.0012981347, 0.0010393162, -0.0..."
3,https://upload.wikimedia.org/wikipedia/commons...,0.873145,"[140, 32, 196, 124]",A person wearing no glasses,0.829759,"{""A person wearing reading glasses"": 0.1326903...","[0.031326413, -0.04205955, -0.056005336, -0.00..."
4,https://upload.wikimedia.org/wikipedia/commons...,0.865377,"[177, 91, 228, 170]",A person wearing no glasses,0.914516,"{""A person wearing reading glasses"": 0.0356861...","[-0.005093807, -0.007854619, -0.03761017, 0.01..."
7,https://upload.wikimedia.org/wikipedia/commons...,0.837970,"[116, 60, 211, 156]",A person wearing no glasses,0.957388,"{""A person wearing reading glasses"": 0.0305605...","[-0.07292417, -0.0046823695, -0.0144624105, 0...."
11,http://upload.wikimedia.org/wikipedia/commons/...,0.847715,"[110, 129, 146, 189]",A person wearing sunglasses,0.864979,"{""A person wearing reading glasses"": 0.0886208...","[0.034851313, 0.0060885483, -0.0068649347, 0.0..."


## 🔍 SQL-like Queries with dask-sql

If you prefer SQL syntax for data queries, you can leverage **dask-sql**. The following example demonstrates how to register your DataFrame as a table and run a SQL query.


In [7]:
# Install dask-sql if necessary (uncomment the next line to install)
# !pip install dask-sql

from dask_sql import Context

# Create a SQL context and register the Dask DataFrame as a table named 'parquet_table'
c = Context()
c.create_table("parquet_table", df)
print("Dask DataFrame registered as table 'parquet_table'.")

# SQL query: Filter images with glasses_confidence > 0.8
query = """
SELECT *
FROM parquet_table
WHERE glasses_confidence > 0.8
"""
result_sql = c.sql(query).compute()

print(f"Total images (SQL query) with glasses_confidence > 0.8: {len(result_sql)}")
result_sql.head()


Dask DataFrame registered as table 'parquet_table'.
Total images (SQL query) with glasses_confidence > 0.8: 2055


,image_url,face_confidence,bbox,glasses_label,glasses_confidence,clip_metadata,clip_embedding
2,https://upload.wikimedia.org/wikipedia/commons...,0.832178,"[127, 78, 182, 133]",A person wearing no glasses,0.953353,"{""A person wearing reading glasses"": 0.0212386...","[-0.01806082, 0.0012981347, 0.0010393162, -0.0..."
3,https://upload.wikimedia.org/wikipedia/commons...,0.873145,"[140, 32, 196, 124]",A person wearing no glasses,0.829759,"{""A person wearing reading glasses"": 0.1326903...","[0.031326413, -0.04205955, -0.056005336, -0.00..."
4,https://upload.wikimedia.org/wikipedia/commons...,0.865377,"[177, 91, 228, 170]",A person wearing no glasses,0.914516,"{""A person wearing reading glasses"": 0.0356861...","[-0.005093807, -0.007854619, -0.03761017, 0.01..."
7,https://upload.wikimedia.org/wikipedia/commons...,0.837970,"[116, 60, 211, 156]",A person wearing no glasses,0.957388,"{""A person wearing reading glasses"": 0.0305605...","[-0.07292417, -0.0046823695, -0.0144624105, 0...."
10,https://upload.wikimedia.org/wikipedia/commons...,0.791787,"[121, 14, 179, 74]",A person wearing no glasses,0.875261,"{""A person wearing reading glasses"": 0.0779443...","[0.009621351, -0.022147348, -0.0028696575, -0...."


### 2. Distributed Data Preprocessing for Model Training
Compute normalization statistics (mean, standard deviation) across a large dataset in a distributed manner to prepare training data.


In [8]:
# Compute summary statistics for glasses_confidence
mean_confidence = df['glasses_confidence'].mean().compute()
std_confidence = df['glasses_confidence'].std().compute()

print("Normalization Statistics for 'glasses_confidence':")
print(f"Mean: {mean_confidence}, Standard Deviation: {std_confidence}")


Normalization Statistics for 'glasses_confidence':
Mean: 0.7355715737688497, Standard Deviation: 0.1630218540532971
